In [ ]:
# KLIG Playground

This notebook imports the KLIG package, loads a ResNet50 model, and lets you experiment with KLIG attribution and Captum baselines.

Instructions:
- Set `IMAGE_PATH` to a local image file.
- Run the cells in order.
- Change the parameters in the playground cells and rerun.
</VSCode.Cell>
<VSCode.Cell language="python">
# Section 1: Import Libraries
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision.transforms as T
from PIL import Image
from torchvision.models import ResNet50_Weights, resnet50

# Ensure the repo root is on sys.path for local imports
ROOT = Path.cwd()
if not (ROOT / "klig").exists():
    ROOT = ROOT / "infocube-main"
sys.path.append(str(ROOT))

from klig.compare.captum_baselines import run_all
from klig.image.attribution import ImageAttributor
from klig.image.stopping import find_sigma_stop
from klig.image.viz import attribution_grid

print("Repo root:", ROOT)
print("KLIG import path OK")
</VSCode.Cell>
<VSCode.Cell language="python">
# Section 2: Define Playground Variables
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMAGE_PATH = ROOT / "example.jpg"  # <- change this to your image path
N_STEPS = 50
N_SAMPLES = 10
SIGMA_FINAL = 1.0 / 256.0
ADAPTIVE_SIGMA = False
IG_STEPS = 50
SG_SAMPLES = 50
CLIP_PCT = 99.0

IMAGENET_TRANSFORM = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print("Device:", DEVICE)
print("Image path:", IMAGE_PATH)
</VSCode.Cell>
<VSCode.Cell language="python">
# Section 3: Create Example Functions

def load_image(path: Path) -> torch.Tensor:
    img = Image.open(path).convert("RGB")
    return IMAGENET_TRANSFORM(img).unsqueeze(0)


def load_model(device: torch.device) -> torch.nn.Module:
    weights = ResNet50_Weights.IMAGENET1K_V2
    model = resnet50(weights=weights).to(device).eval()
    return model


def predict_topk(model: torch.nn.Module, x: torch.Tensor, k: int = 5):
    with torch.no_grad():
        logits = model(x)
        probs = logits.softmax(dim=-1)[0]
        top_probs, top_idx = probs.topk(k)
    labels = ResNet50_Weights.IMAGENET1K_V2.meta["categories"]
    return [(labels[int(i)], float(p)) for i, p in zip(top_idx, top_probs)], top_idx.tolist()


def compute_klig(model: torch.nn.Module, x: torch.Tensor, target: int, n_steps: int, n_samples: int, sigma_final: float, adaptive_sigma: bool):
    if adaptive_sigma:
        sigma_final = find_sigma_stop(model, x, target=target, tau=0.95)
        sigma_final = max(sigma_final, 1.0 / 256.0)
        print(f"Adaptive sigma_final={sigma_final:.6f}")

    attributor = ImageAttributor(
        model=model,
        n_steps=n_steps,
        n_samples=n_samples,
        sigma_final=sigma_final,
        device=DEVICE,
    )
    result = attributor.attribute(x, target=target, show_progress=True)
    return result


def compute_baselines(model: torch.nn.Module, x: torch.Tensor, target: int, ig_steps: int, sg_samples: int, background: torch.Tensor | None = None):
    return run_all(
        model=model,
        x=x,
        target=target,
        ig_steps=ig_steps,
        sg_samples=sg_samples,
        background=background,
    )


def show_results(image: torch.Tensor, top_k_probs: list[tuple[str, float]], rows: list[dict], clip_percentile: float = 99.0):
    fig = attribution_grid(
        image=image.squeeze(0),
        top_k_probs=top_k_probs,
        rows=rows,
        clip_percentile=clip_percentile,
    )
    plt.show()
</VSCode.Cell>
<VSCode.Cell language="python">
# Section 4: Run Example Computations
if not IMAGE_PATH.exists():
    raise FileNotFoundError(f"Change IMAGE_PATH to a valid image file. Current path: {IMAGE_PATH}")

model = load_model(DEVICE)
x = load_image(IMAGE_PATH).to(DEVICE)

# Predict top classes
preds, top_idx = predict_topk(model, x)
print("Top predictions:")
for label, prob in preds:
    print(f"  {label}: {prob:.4f}")

# Use the top predicted class as the explanation target
target_idx = top_idx[0]
print("Explaining class index:", target_idx)

klig_result = compute_klig(
    model=model,
    x=x,
    target=target_idx,
    n_steps=N_STEPS,
    n_samples=N_SAMPLES,
    sigma_final=SIGMA_FINAL,
    adaptive_sigma=ADAPTIVE_SIGMA,
)
print(klig_result)

captum_results = compute_baselines(
    model=model,
    x=x,
    target=target_idx,
    ig_steps=IG_STEPS,
    sg_samples=SG_SAMPLES,
)

rows = [{
    "label": preds[0][0],
    "klig": klig_result,
    "captum": captum_results,
}]

show_results(
    image=x,
    top_k_probs=preds[:5],
    rows=rows,
    clip_percentile=CLIP_PCT,
)
</VSCode.Cell>
<VSCode.Cell language="python">
# Section 5: Experiment with Changes
# Change these parameters and rerun the previous cell to explore behavior.

N_STEPS = 100
N_SAMPLES = 20
SIGMA_FINAL = 1.0 / 128.0
ADAPTIVE_SIGMA = True
IG_STEPS = 100
SG_SAMPLES = 100
CLIP_PCT = 99.0

print("Updated playground parameters:")
print("  N_STEPS", N_STEPS)
print("  N_SAMPLES", N_SAMPLES)
print("  SIGMA_FINAL", SIGMA_FINAL)
print("  ADAPTIVE_SIGMA", ADAPTIVE_SIGMA)
print("  IG_STEPS", IG_STEPS)
print("  SG_SAMPLES", SG_SAMPLES)
